# 🧠 Red Neuronal desde Cero — Comentado y Explicado
**Módulo 4 — Diplomado Deep Learning | Diana Blanco**

Fuente original: https://github.com/rohitg00/ai-engineering-from-scratch

---
## ¿Qué hace este notebook?

Construye una red neuronal **completamente a mano**, sin usar Keras ni TensorFlow.
Solo Python puro + matemáticas.

### ¿Por qué hacerlo así si existe Keras?

> 💡 **Analogía:** Es como aprender a manejar vs. entender el motor.
> Keras es el carro automático — cómodo y rápido.
> Este notebook es abrir el cofre y ver cómo funciona por dentro.
> Cuando uses Keras después, ya sabrás exactamente qué está pasando.

### Conceptos que practica este notebook:
- Qué es una neurona y cómo calcula su salida
- Qué es una capa (layer) y cómo conecta neuronas
- Qué es el **forward pass** (propagación hacia adelante)
- Para qué sirven los **pesos (weights)** y el **sesgo (bias)**
- La función de activación **Sigmoid** y por qué se usa
- Cómo contar parámetros de una arquitectura

---
## 📐 Parte 1: La Función Sigmoid

Antes de las clases, necesitamos una herramienta matemática: la función **sigmoid**.

### ¿Qué hace sigmoid?
Toma cualquier número (desde -∞ hasta +∞) y lo aplasta a un rango entre **0 y 1**.

```
Número muy negativo (-100) → sigmoid → casi 0
Número = 0               → sigmoid → exactamente 0.5
Número muy positivo (+100) → sigmoid → casi 1
```

Fórmula: `σ(x) = 1 / (1 + e^(-x))`

### ¿Por qué se usa?
> 💡 **Analogía:** Imagina un interruptor de luz con dimmer (regulador).
> Un interruptor normal es 0 (apagado) o 1 (encendido) — eso sería una función escalón.
> El dimmer puede estar en cualquier punto intermedio suavemente → eso es sigmoid.
> La suavidad es CRUCIAL porque permite calcular gradientes para el aprendizaje (backprop).

### ¿Qué es el clamp `max(-500, min(500, x))`?
Es una protección numérica. Si `x` fuera 10,000, `e^(-10000)` daría un error de overflow en Python.
El clamp limita x entre -500 y 500 donde sigmoid ya vale prácticamente 0 o 1 de todas formas.

In [1]:
import math
import random

# ── Función Sigmoid ───────────────────────────────────────────────────────────
# σ(x) = 1 / (1 + e^(-x))
# Convierte CUALQUIER número real en un valor entre 0 y 1.
# Esto simula que una neurona "se activa" (→1) o "no se activa" (→0).

def sigmoid(x):
    # Protección contra overflow numérico:
    # e^(-10000) provocaría un error. Limitamos x a [-500, 500]
    # porque sigmoid(-500) ≈ 0 y sigmoid(500) ≈ 1 de todas formas.
    x = max(-500.0, min(500.0, x))
    return 1.0 / (1.0 + math.exp(-x))

# ── Verificar que sigmoid funciona como esperamos ─────────────────────────────
print("Verificando sigmoid:")
print(f"  sigmoid(-100) = {sigmoid(-100):.6f}  ← casi 0 (neurona inactiva)")
print(f"  sigmoid(0)    = {sigmoid(0):.6f}  ← exactamente 0.5 (neutro)")
print(f"  sigmoid(100)  = {sigmoid(100):.6f}  ← casi 1 (neurona activa)")
print(f"  sigmoid(2)    = {sigmoid(2):.6f}  ← entre 0.5 y 1")

Verificando sigmoid:
  sigmoid(-100) = 0.000000  ← casi 0 (neurona inactiva)
  sigmoid(0)    = 0.500000  ← exactamente 0.5 (neutro)
  sigmoid(100)  = 1.000000  ← casi 1 (neurona activa)
  sigmoid(2)    = 0.880797  ← entre 0.5 y 1


---
## 🧱 Parte 2: La Clase Layer (Capa de Neuronas)

Una **capa** es un grupo de neuronas que trabajan en paralelo.
Cada neurona en la capa recibe TODAS las entradas y produce UNA salida.

### ¿Qué hace una neurona matemáticamente?
```
Entradas:  x1, x2, ..., xn
Pesos:     w1, w2, ..., wn   ← qué tan importante es cada entrada
Sesgo:     b                 ← ajuste independiente de las entradas

Paso 1 — Suma ponderada:  z = (w1·x1) + (w2·x2) + ... + (wn·xn) + b
Paso 2 — Activación:      salida = sigmoid(z)
```

> 💡 **Analogía:** Imagina que eres un crítico de películas.
> Recibes señales: actuación (x1), guión (x2), efectos (x3).
> Cada criterio tiene un peso para ti: actuación (w1=0.5), guión (w2=0.4), efectos (w3=0.1).
> Sumas todo, y el sesgo (b) es tu "mood" del día — ajusta tu calificación base.
> Si la suma supera tu umbral → das buena crítica (salida → 1).

### ¿Qué son los pesos y el sesgo?

| Concepto | Qué es | Analogía |
|---|---|---|
| **Peso (weight)** | Importancia de cada conexión | Volumen de cada canal de audio |
| **Sesgo (bias)** | Ajuste independiente | El brillo base de una pantalla antes de ajustar contraste |
| **Inicialización aleatoria** | Pesos al azar al crear la capa | Empezar el dimmer en posición aleatoria antes de calibrar |

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CLASE: Layer (Capa)
# Representa un grupo de neuronas que procesan las mismas entradas en paralelo.
# ═══════════════════════════════════════════════════════════════════════════════

class Layer:
    
    def __init__(self, n_inputs, n_neurons, weights=None, biases=None):
        """
        Constructor: define la forma de la capa.
        
        n_inputs:  cuántas señales recibe cada neurona (= tamaño de la capa anterior)
        n_neurons: cuántas neuronas tiene esta capa (= tamaño de la salida)
        weights:   lista de pesos precargados (opcional). Si no se dan, se inicializan al azar.
        biases:    lista de sesgos precargados (opcional). Si no se dan, se inicializan en 0.
        
        Estructura de self.weights:
            self.weights[i] = lista de pesos DE la neurona i
            self.weights[i][j] = peso de la conexión desde la entrada j HACIA la neurona i
            
        Ejemplo para Layer(n_inputs=2, n_neurons=3):
            self.weights = [
                [w00, w01],   ← pesos de la neurona 0 (recibe 2 entradas)
                [w10, w11],   ← pesos de la neurona 1
                [w20, w21],   ← pesos de la neurona 2
            ]
            self.biases = [b0, b1, b2]  ← un sesgo por neurona
        """
        
        # Si se proveen pesos externos, usarlos (modo "pesos a mano")
        # Si no, inicializar ALEATORIAMENTE en rango [-1, 1]
        # La inicialización aleatoria es importante: si todos los pesos fueran
        # iguales (ej. todos 0), todas las neuronas aprenderían lo mismo → inútil.
        if weights is not None:
            self.weights = weights
        else:
            # Crea una matriz de pesos: n_neurons filas × n_inputs columnas
            # random.uniform(-1, 1) da un número decimal entre -1 y 1
            self.weights = [
                [random.uniform(-1, 1) for _ in range(n_inputs)]  # pesos de UNA neurona
                for _ in range(n_neurons)                          # repite para cada neurona
            ]
        
        # Sesgos: uno por neurona, inicializados en 0 si no se dan
        # (es práctica común iniciar biases en 0, no al azar)
        if biases is not None:
            self.biases = biases
        else:
            self.biases = [0.0] * n_neurons  # [0.0, 0.0, ..., 0.0]

    def forward(self, inputs):
        """
        Forward pass de la capa: procesa las entradas y produce salidas.
        Cada neurona hace: salida = sigmoid(suma_ponderada + sesgo)
        
        inputs: lista de valores de entrada (viene de la capa anterior o del dataset)
        retorna: lista de activaciones, una por neurona de esta capa
        """
        
        # Guardamos la entrada (útil para backprop más adelante)
        self.last_input = inputs
        self.last_output = []
        
        # Procesamos cada neurona de la capa una por una
        for neuron_idx in range(len(self.weights)):
            
            # ── Suma ponderada: z = Σ(w_i · x_i) + b ──────────────────────
            # zip(weights, inputs) empareja: [(w0,x0), (w1,x1), ...]
            # sum(...) suma todos los productos w·x
            z = sum(
                w * x
                for w, x in zip(self.weights[neuron_idx], inputs)
            )
            z += self.biases[neuron_idx]  # sumamos el sesgo de esta neurona
            
            # ── Función de activación ───────────────────────────────────────
            # Aplica sigmoid para convertir z (cualquier número) en [0, 1]
            self.last_output.append(sigmoid(z))
        
        return self.last_output


# ── Demostración manual de una capa ──────────────────────────────────────────
print("Ejemplo: Layer(n_inputs=2, n_neurons=3)")
capa_demo = Layer(n_inputs=2, n_neurons=3)
print(f"  Pesos (3 neuronas × 2 entradas): {capa_demo.weights}")
print(f"  Sesgos: {capa_demo.biases}")
salida_demo = capa_demo.forward([0.5, 0.8])
print(f"  Salida con entradas [0.5, 0.8]: {[f'{v:.4f}' for v in salida_demo]}")
print(f"  → 3 salidas porque hay 3 neuronas")

Ejemplo: Layer(n_inputs=2, n_neurons=3)
  Pesos (3 neuronas × 2 entradas): [[-0.7144361214994275, 0.5525223169508384], [0.3691380575733647, -0.5644637161797126], [0.5514205421852802, -0.8048087400346327]]
  Sesgos: [0.0, 0.0, 0.0]
  Salida con entradas [0.5, 0.8]: ['0.5212', '0.4336', '0.4090']
  → 3 salidas porque hay 3 neuronas


---
## 🏗️ Parte 3: La Clase Network (Red Completa)

Una **red neuronal** es simplemente una secuencia de capas encadenadas.
La salida de una capa es la entrada de la siguiente.

```
Datos → [Capa 1] → [Capa 2] → [Capa 3] → Predicción
  X        h1         h2         ŷ
```

> 💡 **Analogía:** Es como una línea de ensamblaje.
> El material bruto (datos) entra, cada estación (capa) lo transforma,
> y al final sale el producto terminado (predicción).
>
> La Capa 1 detecta patrones simples ("¿hay una curva aquí?").
> La Capa 2 combina esos patrones ("curva + línea = ojo").
> La Capa 3 toma la decisión final ("esto es un gato").

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CLASE: Network (Red Neuronal Completa)
# Encadena múltiples capas y ejecuta el forward pass completo.
# ═══════════════════════════════════════════════════════════════════════════════

class Network:
    
    def __init__(self, layers):
        """
        Constructor: recibe una lista de objetos Layer ya creados.
        El orden importa: la capa en layers[0] es la primera en procesar.
        
        IMPORTANTE: n_inputs de cada capa debe coincidir con n_neurons
        de la capa anterior. Si no, habrá error al hacer el forward pass.
        
        Ejemplo arquitectura 2-4-1:
            layers = [
                Layer(n_inputs=2, n_neurons=4),  ← capa oculta: recibe 2, produce 4
                Layer(n_inputs=4, n_neurons=1),  ← capa salida: recibe 4, produce 1
            ]
        """
        self.layers = layers

    def forward(self, inputs):
        """
        Forward pass completo: pasa los datos por TODAS las capas en orden.
        
        inputs: datos de entrada (lista de números)
        retorna: salida de la ÚLTIMA capa (la predicción)
        """
        current = inputs  # empezamos con los datos originales
        
        for layer in self.layers:
            # La salida de esta capa se convierte en la entrada de la siguiente
            # Red 2-4-1: inputs[2] → Layer1.forward → [4 valores] → Layer2.forward → [1 valor]
            current = layer.forward(current)
        
        return current  # = salida de la última capa = predicción

    def count_parameters(self):
        """
        Cuenta el total de parámetros entrenables (pesos + sesgos).
        
        ¿Para qué sirve? Para saber qué tan 'grande' es una red.
        Más parámetros = más capacidad de aprender, pero también más lenta
        y con más riesgo de overfitting.
        
        Fórmula para una capa: (n_inputs × n_neurons) pesos + n_neurons sesgos
        Ejemplo Layer(2, 3):  (2 × 3) + 3 = 9 parámetros
        """
        total = 0
        for layer in self.layers:
            # Contar pesos: recorremos cada neurona y sumamos cuántos pesos tiene
            for neuron_weights in layer.weights:
                total += len(neuron_weights)  # = n_inputs de esa capa
            # Contar sesgos: uno por neurona
            total += len(layer.biases)        # = n_neurons de esa capa
        return total


print("Clase Network definida ✓")
print()
print("Verificación de count_parameters:")
net_test = Network([
    Layer(n_inputs=2, n_neurons=3),   # pesos: 2×3=6, sesgos: 3 → total: 9
    Layer(n_inputs=3, n_neurons=1),   # pesos: 3×1=3, sesgos: 1 → total: 4
])
print(f"  Red 2-3-1: {net_test.count_parameters()} parámetros (esperado: 13)")

Clase Network definida ✓

Verificación de count_parameters:
  Red 2-3-1: 13 parámetros (esperado: 13)


---
## 🎯 DEMO 1: El Problema XOR

### ¿Qué es XOR?
XOR (OR exclusivo) es una operación lógica:

| x1 | x2 | Salida XOR |
|---|---|---|
| 0 | 0 | **0** (ninguno activo → 0) |
| 0 | 1 | **1** (solo uno activo → 1) |
| 1 | 0 | **1** (solo uno activo → 1) |
| 1 | 1 | **0** (ambos activos → 0) |

### ¿Por qué XOR es famoso en redes neuronales?
> 💡 **Contexto histórico:** En 1969, Minsky y Papert demostraron que un perceptrón simple
> (una sola neurona) NO puede resolver XOR — los puntos no son linealmente separables.
> Esto "mató" el interés en redes neuronales por años.
> La solución: agregar capas ocultas. Una red 2-2-1 SÍ puede resolverlo.

### ¿Qué significa "pesos a mano" (hand-tuned)?
Alguien ya calculó los pesos correctos manualmente para que la red resuelva XOR.
En la práctica real, el entrenamiento (backprop) encontraría estos valores automáticamente.
Aquí los fijamos a mano para demostrar que la arquitectura puede resolver el problema.

In [ ]:
print("=" * 60)
print("DEMO 1: XOR resuelto con red 2-2-1 (pesos a mano)")
print("=" * 60)

# ── Capa oculta: 2 entradas, 2 neuronas ──────────────────────────────────────
# Los pesos NO son aleatorios aquí — ya sabemos cuáles funcionan para XOR.
# 
# Neurona 0: pesos [20, 20], sesgo -10
#   → Se activa cuando AMBAS entradas son 1 (suma = 40 - 10 = 30 → sigmoid → ~1)
#   → No se activa si ambas son 0 (suma = 0 - 10 = -10 → sigmoid → ~0)
#   → Se activa a medias con una entrada (suma = 20 - 10 = 10 → sigmoid → ~1)
#   → Básicamente implementa: ¿al menos una entrada es 1? (OR lógico)
#
# Neurona 1: pesos [-20, -20], sesgo 30
#   → Se activa cuando NINGUNA entrada es 1 (suma = 0 + 30 = 30 → sigmoid → ~1)
#   → No se activa cuando ambas son 1 (suma = -40 + 30 = -10 → sigmoid → ~0)
#   → Básicamente implementa: ¿NINGUNA entrada es 1? (NOR lógico, invertido)
#   → Juntas las 2 neuronas: detectan el caso especial de "ambas = 1"

hidden = Layer(
    n_inputs=2,
    n_neurons=2,
    weights=[[20.0, 20.0],    # neurona 0: pesos positivos → OR
             [-20.0, -20.0]], # neurona 1: pesos negativos → NOR
    biases=[-10.0, 30.0],     # sesgos ajustados para los umbrales correctos
)

# ── Capa de salida: 2 entradas (viene de hidden), 1 neurona ──────────────────
# Recibe las activaciones de las 2 neuronas ocultas y toma la decisión final.
# Pesos [20, 20], sesgo -30
#
# Lógica:
#   Entrada [0,0]: hidden produce [~0, ~1] → suma = 0×20 + 1×20 - 30 = -10 → ~0 ✓
#   Entrada [0,1]: hidden produce [~1, ~1] → suma = 1×20 + 1×20 - 30 = 10  → ~1 ✓
#   Entrada [1,1]: hidden produce [~1, ~0] → suma = 1×20 + 0×20 - 30 = -10 → ~0 ✓

output = Layer(
    n_inputs=2,
    n_neurons=1,
    weights=[[20.0, 20.0]],
    biases=[-30.0],
)

# ── Crear la red encadenando las capas ────────────────────────────────────────
xor_net = Network([hidden, output])

# ── Dataset XOR completo ──────────────────────────────────────────────────────
# Cada elemento: (lista_de_entradas, salida_esperada)
xor_data = [
    ([0, 0], 0),  # 0 XOR 0 = 0
    ([0, 1], 1),  # 0 XOR 1 = 1
    ([1, 0], 1),  # 1 XOR 0 = 1
    ([1, 1], 0),  # 1 XOR 1 = 0
]

# ── Probar la red con cada combinación ───────────────────────────────────────
all_correct = True
for inputs, expected in xor_data:
    result = xor_net.forward(inputs)  # forward pass: obtiene probabilidad
    
    # Decisión: si la probabilidad >= 0.5, predecimos clase 1, si no, clase 0
    # Esto se llama "umbral de decisión" o "threshold"
    predicted = 1 if result[0] >= 0.5 else 0
    
    status = "✅ OK" if predicted == expected else "❌ WRONG"
    if predicted != expected:
        all_correct = False
    
    print(f"  Entradas {inputs} → valor raw: {result[0]:.6f} "
          f"→ predicción: {predicted} | esperado: {expected} {status}")

print(f"\n¿XOR resuelto? {all_correct}")
print(f"Total de parámetros en la red: {xor_net.count_parameters()}")
# 9 parámetros = hidden(4 pesos + 2 sesgos) + output(2 pesos + 1 sesgo)

DEMO 1: XOR resuelto con red 2-2-1 (pesos a mano)
  Entradas [0, 0] → valor raw: 0.000045 → predicción: 0 | esperado: 0 ✅ OK
  Entradas [0, 1] → valor raw: 0.999955 → predicción: 1 | esperado: 1 ✅ OK
  Entradas [1, 0] → valor raw: 0.999955 → predicción: 1 | esperado: 1 ✅ OK
  Entradas [1, 1] → valor raw: 0.000045 → predicción: 0 | esperado: 0 ✅ OK

¿XOR resuelto? True
Total de parámetros en la red: 9


: 

---
## ⭕ DEMO 2: Clasificar puntos dentro/fuera de un círculo

### El problema:
Tenemos puntos (x, y) en un plano 2D dentro del cuadrado [-1, 1] × [-1, 1].
Un punto está **dentro del círculo** si su distancia al centro es menor a 0.5:

```
x² + y² < 0.25  →  dentro del círculo  (radio = 0.5, radio² = 0.25)
x² + y² ≥ 0.25  →  fuera del círculo
```

### ¿Por qué los pesos aleatorios dan mala accuracy?
> 💡 Si una red no ha sido entrenada, predice casi al azar.
> Solo ~17.5% de los puntos están dentro del círculo,
> así que si la red predice siempre "fuera", acertaría ~82.5%.
> Con pesos aleatorios no hace ni eso — demuestra que **sin entrenar, la arquitectura sola no sirve**.

In [ ]:
print("=" * 60)
print("DEMO 2: Clasificar puntos dentro/fuera de un círculo")
print("=" * 60)

# ── Generar dataset ────────────────────────────────────────────────────────────
random.seed(42)  # semilla fija: garantiza que siempre generemos los MISMOS puntos
                 # sin esto, cada ejecución daría puntos diferentes → no reproducible

data = []
for _ in range(200):              # generamos 200 puntos
    x = random.uniform(-1, 1)     # coordenada x: número decimal entre -1 y 1
    y = random.uniform(-1, 1)     # coordenada y: número decimal entre -1 y 1
    
    # Etiqueta: 1 si está DENTRO del círculo de radio 0.5, 0 si está FUERA
    # Teorema de Pitágoras: distancia² = x² + y²
    # Si distancia² < 0.5² (= 0.25) → está dentro
    label = 1 if (x * x + y * y) < 0.25 else 0
    
    data.append(([x, y], label))  # guardar punto y su etiqueta

# Estadísticas del dataset
inside_count  = sum(1 for _, label in data if label == 1)
outside_count = len(data) - inside_count
print(f"  Dataset: {len(data)} puntos ({inside_count} dentro, {outside_count} fuera)")
print(f"  Balance: {inside_count/len(data)*100:.1f}% dentro / {outside_count/len(data)*100:.1f}% fuera")
print(f"  → Dataset desbalanceado: pocos puntos caben en un círculo pequeño")

# ── Crear red con pesos ALEATORIOS ────────────────────────────────────────────
random.seed(7)   # semilla diferente para los pesos (también reproducible)
circle_net = Network([
    Layer(n_inputs=2, n_neurons=8),   # capa oculta: 2 entradas (x,y) → 8 neuronas
    Layer(n_inputs=8, n_neurons=1),   # capa salida: 8 → 1 (probabilidad dentro/fuera)
])

# ── Evaluar accuracy con pesos sin entrenar ───────────────────────────────────
correct = 0
for inputs, expected in data:
    result    = circle_net.forward(inputs)
    predicted = 1 if result[0] >= 0.5 else 0
    if predicted == expected:
        correct += 1

print(f"\n  Accuracy con pesos ALEATORIOS: {correct}/{len(data)} ({100 * correct / len(data):.1f}%)")
print(f"  Parámetros de la red: {circle_net.count_parameters()}")
print(f"\n  ⚠️  Con pesos aleatorios la red no sabe nada.")
print(f"  Para que aprenda se necesita entrenamiento (backprop).")
print(f"  Eso lo veremos en las siguientes clases del módulo.")

# ── Visualizar los puntos (bonus) ─────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(6, 6))
for (x, y), label in data:
    color = '#9b59b6' if label == 1 else '#95a5a6'  # morado=dentro, gris=fuera
    ax.scatter(x, y, c=color, s=15, alpha=0.7)

# Dibujar el círculo real
circulo = plt.Circle((0, 0), 0.5, fill=False, color='#e74c3c', linewidth=2, label='Frontera real')
ax.add_patch(circulo)
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect('equal')
ax.set_title('Dataset: puntos dentro/fuera del círculo', fontsize=12)
ax.legend(['Dentro (morado)', 'Fuera (gris)', 'Frontera real'])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

DEMO 2: Clasificar puntos dentro/fuera de un círculo
  Dataset: 200 puntos (35 dentro, 165 fuera)
  Balance: 17.5% dentro / 82.5% fuera
  → Dataset desbalanceado: pocos puntos caben en un círculo pequeño

  Accuracy con pesos ALEATORIOS: 35/200 (17.5%)
  Parámetros de la red: 33

  ⚠️  Con pesos aleatorios la red no sabe nada.
  Para que aprenda se necesita entrenamiento (backprop).
  Eso lo veremos en las siguientes clases del módulo.


---
## 🔍 DEMO 3: Ver el forward pass por dentro

Aquí vemos exactamente cómo se transforma la información al pasar por la red.
Cada capa va *representando* el problema de forma diferente hasta que
la última capa puede tomar la decisión correcta.

> 💡 **Analogía:** Es como un proceso de traducción.
> Entrada [0, 1] en "idioma datos crudos" → capa oculta lo convierte a [0.9999, 0.9999] 
> en "idioma representación intermedia" → capa salida lo convierte a 0.9999 en
> "idioma predicción".

In [ ]:
print("=" * 60)
print("DEMO 3: Internos del forward pass en XOR")
print("=" * 60)

for inputs, expected in xor_data:
    # forward() actualiza layer.last_output en cada capa
    xor_net.forward(inputs)
    
    # Acceder a los valores intermedios guardados en cada capa
    # xor_net.layers[0] = capa oculta
    # xor_net.layers[1] = capa de salida
    h = xor_net.layers[0].last_output   # representación intermedia (oculta)
    o = xor_net.layers[1].last_output   # predicción final
    
    prediccion = '1' if o[0] >= 0.5 else '0'
    correcto   = '✅' if int(prediccion) == expected else '❌'
    
    print(f"\n  Entrada: {inputs}")
    print(f"    Capa oculta  → [{h[0]:.6f}, {h[1]:.6f}]  ← representación transformada")
    print(f"    Capa salida  → {o[0]:.6f}  → clase: {prediccion} | esperado: {expected} {correcto}")

print()
print("Observaciones:")
print("  - [0,0]: oculta da [~0, ~1] → la segunda neurona oculta se activa sola")
print("  - [0,1] y [1,0]: oculta da [~1, ~1] → ambas neuronas activas")
print("  - [1,1]: oculta da [~1, ~0] → solo la primera neurona activa")
print()
print("  La capa oculta separó el espacio de forma que la salida puede")
print("  distinguir los casos imposibles de separar linealmente.")

---
## 📊 DEMO 4: Contar parámetros de arquitecturas clásicas

### ¿Por qué importa contar parámetros?

| Parámetros | Implicación |
|---|---|
| Pocos (< 1k) | Red simple, puede aprender patrones básicos |
| Medios (100k–1M) | Tareas de clasificación de imágenes |
| Muchos (> 1B) | Modelos de lenguaje (GPT-4 ~1.8 trillones) |

### Fórmula para calcular:
```
Para una capa que recibe N entradas y tiene M neuronas:
Parámetros = (N × M) pesos + M sesgos

Para una red completa: sumar todos los parámetros de cada capa

Ejemplo red 784-256-128-10 (MNIST):
Capa 1: (784 × 256) + 256  = 200,960
Capa 2: (256 × 128) + 128  =  32,896
Capa 3: (128 × 10)  + 10   =   1,290
TOTAL                       = 235,146 ✓
```

In [ ]:
print("=" * 60)
print("DEMO 4: Conteo de parámetros en arquitecturas clásicas")
print("=" * 60)

# Lista de arquitecturas a comparar:
# Cada tupla: (nombre_descriptivo, [tamaños_de_capas])
# Los tamaños incluyen la capa de entrada → capas del modelo son los saltos
architectures = [
    ("2-3-1 (este notebook)",        [2, 3, 1]),           # red chiquita
    ("2-8-1 (círculo)",              [2, 8, 1]),           # un poco más capaz
    ("784-256-128-10 (MNIST base)",  [784, 256, 128, 10]), # reconocimiento de dígitos
    ("784-512-256-128-10 (MNIST++)", [784, 512, 256, 128, 10]), # más profunda
]

print(f"\n{'Arquitectura':<40} {'Parámetros':>15}")
print("-" * 57)

for name, sizes in architectures:
    layers_list = []
    
    # Crear las capas: para una red [2, 3, 1] hay 2 capas:
    # - Layer(n_inputs=2, n_neurons=3)   ← conecta entrada con capa oculta
    # - Layer(n_inputs=3, n_neurons=1)   ← conecta capa oculta con salida
    # El range(1, len(sizes)) va de índice 1 al final: usamos pares [i-1, i]
    for i in range(1, len(sizes)):
        layers_list.append(Layer(n_inputs=sizes[i-1], n_neurons=sizes[i]))
    
    net = Network(layers_list)
    params = net.count_parameters()
    print(f"  {name:<38} {params:>12,} parámetros")

print()
print("  Nota: MNIST tiene imágenes de 28×28 = 784 píxeles (entrada)")
print("  y 10 clases (dígitos del 0 al 9) en la salida.")

---
## 🏋️ Ejercicios

Los siguientes ejercicios exploran variaciones de la red base.

In [ ]:
# ── Ejercicio 1: Red 2-4-2-1 con dos capas ocultas ───────────────────────────
# Objetivo: ver cómo se transforma la representación en cada capa oculta

print("Ejercicio 1: Red 2-4-2-1 — dos capas ocultas")
print("-" * 50)

random.seed(99)
red_profunda = Network([
    Layer(n_inputs=2, n_neurons=4),  # capa oculta 1: transforma 2 → 4 representaciones
    Layer(n_inputs=4, n_neurons=2),  # capa oculta 2: comprime 4 → 2 representaciones
    Layer(n_inputs=2, n_neurons=1),  # capa salida:   decide con 2 → 1
])

print(f"Parámetros totales: {red_profunda.count_parameters()}")
print(f"  Capa 1 (2→4): {2*4 + 4} = 12")
print(f"  Capa 2 (4→2): {4*2 + 2} = 10")
print(f"  Capa 3 (2→1): {2*1 + 1} =  3")
print()

for inputs, expected in xor_data:
    red_profunda.forward(inputs)
    h1 = red_profunda.layers[0].last_output  # representación capa 1
    h2 = red_profunda.layers[1].last_output  # representación capa 2
    o  = red_profunda.layers[2].last_output  # salida final
    
    predicted = 1 if o[0] >= 0.5 else 0
    print(f"  Entrada {inputs}:")
    print(f"    Oculta-1 (4): {[f'{v:.3f}' for v in h1]}")
    print(f"    Oculta-2 (2): {[f'{v:.3f}' for v in h2]}")
    print(f"    Salida   (1): {o[0]:.6f} → {predicted} (esperado: {expected})")

print()
print("⚠️  Con pesos aleatorios los resultados son incorrectos.")
print("   La red tiene CAPACIDAD para resolver XOR, pero necesita entrenamiento.")

In [ ]:
# ── Ejercicio 2: Cambiar tamaño de capa oculta en el clasificador de círculo ──
# Objetivo: ¿cambia el rango de salida con más/menos neuronas?

print("Ejercicio 2: Impacto del tamaño de capa oculta")
print("-" * 50)

for n_hidden in [2, 8, 32]:
    random.seed(42)
    net = Network([
        Layer(n_inputs=2, n_neurons=n_hidden),
        Layer(n_inputs=n_hidden, n_neurons=1),
    ])
    
    salidas = [net.forward([x, y])[0] for ([x, y], _) in data]
    
    correct = sum(
        1 for ([x, y], label) in data
        if (1 if net.forward([x, y])[0] >= 0.5 else 0) == label
    )
    
    print(f"  n_hidden={n_hidden:2d}: "
          f"params={net.count_parameters():4d} | "
          f"salida min={min(salidas):.3f} max={max(salidas):.3f} | "
          f"accuracy={correct/len(data)*100:.1f}%")

print()
print("Observación: el rango [0,1] no cambia (sigmoid siempre lo limita).")
print("Pero con más neuronas el modelo tiene MÁS CAPACIDAD para aprender fronteras complejas.")
print("Con pesos aleatorios la accuracy sigue siendo mala — falta entrenamiento.")

---
## 🎨 Ejercicio 4: Red 3-4-4-2 — Clasificador de Colores RGB

### El problema:
Una imagen digital representa cada píxel con tres valores: **R** (rojo), **G** (verde), **B** (azul),
cada uno normalizado en el rango **[0, 1]**.

La red recibe un color como entrada `[R, G, B]` y produce **dos salidas**,
una por clase (por ejemplo: ¿es un color "cálido" o "frío"?).

### Arquitectura 3-4-4-2:
```
Entrada  →  Oculta-1  →  Oculta-2  →  Salida
  3 (RGB)      4             4           2 (clases)
```

| Capa | n_inputs | n_neurons | Función |
|---|---|---|---|
| Oculta 1 | 3 | 4 | Detecta combinaciones de canales RGB |
| Oculta 2 | 4 | 4 | Refina la representación intermedia |
| Salida   | 4 | 2 | Probabilidad de pertenecer a cada clase |

> 💡 **Nota:** Con pesos aleatorios las salidas no tienen sentido semántico todavía.
> El ejercicio muestra la **estructura** del forward pass, no un clasificador entrenado.
> Para que la red realmente clasifique colores se necesitaría un dataset etiquetado y entrenamiento.

In [ ]:
# ── Ejercicio 4: Red 3-4-4-2 para clasificar colores RGB ─────────────────────
# Entrada: [R, G, B] con valores entre 0.0 y 1.0
# Salida: [prob_clase0, prob_clase1]  ← dos probabilidades (ej: cálido vs frío)

print("Ejercicio 4: Red 3-4-4-2 — Clasificador de Colores RGB")
print("-" * 55)

random.seed(77)
red_colores = Network([
    Layer(n_inputs=3, n_neurons=4),  # oculta 1: [R,G,B] → 4 features
    Layer(n_inputs=4, n_neurons=4),  # oculta 2: 4 → 4 (refina la representación)
    Layer(n_inputs=4, n_neurons=2),  # salida:   4 → 2 probabilidades
])

print(f"Parámetros totales: {red_colores.count_parameters()}")
print(f"  Capa 1 (3→4): {3*4 + 4} = 16")
print(f"  Capa 2 (4→4): {4*4 + 4} = 20")
print(f"  Capa 3 (4→2): {4*2 + 2} = 10")
print()

# ── Colores de prueba (R, G, B) normalizados a [0, 1] ────────────────────────
# Formato: (nombre_del_color, [R, G, B])
colores_prueba = [
    ("Rojo puro     ", [1.0, 0.0, 0.0]),
    ("Verde puro    ", [0.0, 1.0, 0.0]),
    ("Azul puro     ", [0.0, 0.0, 1.0]),
    ("Blanco        ", [1.0, 1.0, 1.0]),
    ("Negro         ", [0.0, 0.0, 0.0]),
    ("Gris medio    ", [0.5, 0.5, 0.5]),
    ("Amarillo      ", [1.0, 1.0, 0.0]),
    ("Cian          ", [0.0, 1.0, 1.0]),
    ("Magenta       ", [1.0, 0.0, 1.0]),
    ("Morado pastel ", [0.7, 0.5, 0.9]),  # ~Lavanda
]

print(f"{'Color':<18} {'RGB':^20} {'Oculta-1':^30} {'Salida [c0, c1]':^22} {'Clase'}")
print("-" * 105)

for nombre, rgb in colores_prueba:
    # Forward pass completo
    red_colores.forward(rgb)
    
    # Recuperar representaciones intermedias
    h1 = red_colores.layers[0].last_output   # salida capa oculta 1 (4 valores)
    h2 = red_colores.layers[1].last_output   # salida capa oculta 2 (4 valores)
    out = red_colores.layers[2].last_output  # salida final (2 probabilidades)
    
    # Decisión: la clase es el índice de la probabilidad más alta
    # (equivalente a argmax — en Keras sería np.argmax(output))
    clase_predicha = 0 if out[0] >= out[1] else 1
    
    rgb_str  = f"[{rgb[0]:.1f}, {rgb[1]:.1f}, {rgb[2]:.1f}]"
    h1_str   = f"[{', '.join(f'{v:.2f}' for v in h1)}]"
    out_str  = f"[{out[0]:.4f}, {out[1]:.4f}]"
    
    print(f"  {nombre} {rgb_str:^18} {h1_str:^32} {out_str:^22} → Clase {clase_predicha}")

print()
print("Observaciones con pesos ALEATORIOS:")
print("  1. Las salidas siempre están en [0, 1] gracias a sigmoid.")
print("  2. La red asigna clases, pero sin sentido — no ha sido entrenada.")
print("  3. Oculta-1 transforma [R,G,B] → 4 valores: mezclas de los 3 canales.")
print("  4. Con entrenamiento, la red aprendería que rojo/amarillo = cálido,")
print("     azul/cian = frío, etc.")
print()

# ── Visualización: los colores de prueba ─────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, len(colores_prueba), figsize=(14, 2))
fig.suptitle('Colores de entrada a la red 3-4-4-2', fontsize=11, y=1.05)

for ax, (nombre, rgb) in zip(axes, colores_prueba):
    ax.add_patch(mpatches.Rectangle((0, 0), 1, 1, color=rgb))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(nombre.strip(), fontsize=7, pad=3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Ejercicio 5: Reemplazar sigmoid con 'leaky step' ─────────────────────────
# Objetivo: entender por qué sigmoid (suave) es mejor que funciones con cortes bruscos

print("Ejercicio 5: Leaky Step vs. Sigmoid")
print("-" * 50)

def leaky_step(z):
    """
    Función de activación 'leaky step':
    - Si z < 0: devuelve 0.01 * z  (pendiente muy pequeña, casi 0)
    - Si z >= 0: devuelve 1.0      (salida constante)
    
    El problema: tiene una DISCONTINUIDAD en z=0.
    La derivada casi siempre es 0 → el gradiente no fluye → no aprende.
    """
    return 0.01 * z if z < 0 else 1.0

# Modificar temporalmente la función sigmoid de las capas del xor_net
# Para demostrar el concepto, creamos una capa manual con leaky_step

class LayerLeaky(Layer):
    """Versión de Layer que usa leaky_step en vez de sigmoid."""
    def forward(self, inputs):
        self.last_input = inputs
        self.last_output = []
        for neuron_idx in range(len(self.weights)):
            z = sum(w * x for w, x in zip(self.weights[neuron_idx], inputs))
            z += self.biases[neuron_idx]
            self.last_output.append(leaky_step(z))  # ← leaky_step en vez de sigmoid
        return self.last_output

# Red XOR con los mismos pesos a mano pero usando leaky_step
hidden_leaky = LayerLeaky(
    n_inputs=2, n_neurons=2,
    weights=[[20.0, 20.0], [-20.0, -20.0]],
    biases=[-10.0, 30.0]
)
output_leaky = LayerLeaky(
    n_inputs=2, n_neurons=1,
    weights=[[20.0, 20.0]],
    biases=[-30.0]
)
xor_leaky = Network([hidden_leaky, output_leaky])

print("  Con sigmoid (suave):   ", end="")
resultados_sig = []
for inputs, expected in xor_data:
    r = xor_net.forward(inputs)
    resultados_sig.append(1 if r[0] >= 0.5 else 0)
print(resultados_sig, "← espera [0,1,1,0]")

print("  Con leaky_step (brusca):", end=" ")
resultados_ls = []
for inputs, expected in xor_data:
    r = xor_leaky.forward(inputs)
    resultados_ls.append(1 if r[0] >= 0.5 else 0)
print(resultados_ls, "← ¿mismo resultado?")

print()
print("  ¿Por qué sigmoid es preferida para ENTRENAMIENTO?")
print("  Sigmoid es suave y diferenciable en TODO punto.")
print("  Leaky step tiene derivada ≈ 0 casi siempre → gradiente = 0 → la red no aprende.")
print("  Con pesos a mano puede funcionar, pero durante backprop sigmoid es esencial.")

---
## 📖 Glosario de este notebook

Términos específicos usados en este ejercicio, en el orden en que aparecen:

| Término | Definición |
|---|---|
| **Clase (class)** | Plantilla en Python para crear objetos con atributos y métodos. `Layer` y `Network` son clases. |
| **Constructor (`__init__`)** | Método especial que se llama automáticamente al crear un objeto. Inicializa sus atributos. |
| **`self`** | Referencia al objeto actual. `self.weights` = los pesos de *esta* capa específica. |
| **Inicialización aleatoria** | Asignar pesos al azar antes de entrenar. Necesaria para que las neuronas aprendan cosas diferentes. |
| **`random.uniform(a, b)`** | Genera un número decimal aleatorio entre a y b. |
| **`random.seed(n)`** | Fija el generador aleatorio → mismo seed = mismos números = reproducibilidad. |
| **`zip(lista1, lista2)`** | Une dos listas en pares: `zip([1,2], ['a','b'])` → `[(1,'a'), (2,'b')]`. |
| **`sum(expresion for ... in ...)`** | Suma el resultado de una expresión aplicada a cada elemento. Equivalente a un for loop + suma. |
| **Forward pass** | Calcular la salida de la red pasando los datos de entrada → capa 1 → capa 2 → ... → salida. |
| **`last_input` / `last_output`** | Atributos que guardan los valores del último forward pass. Necesarios para backprop después. |
| **Suma ponderada** | `z = Σ(w_i · x_i) + b`. El valor que entra a la función de activación. |
| **Umbral de decisión** | El valor que decide la clase: si salida ≥ 0.5 → clase 1, si no → clase 0. |
| **`__main__`** | Bloque `if __name__ == '__main__':` ejecuta código solo cuando el archivo se corre directamente (no cuando se importa). En notebooks se puede ignorar o reemplazar por celdas. |
| **XOR** | Operación lógica: devuelve 1 solo cuando exactamente UNA de las dos entradas es 1. No es linealmente separable. |
| **Linealmente separable** | Un problema donde se puede separar las clases con una línea recta (en 2D) o hiperplano. XOR NO lo es. |
| **Hand-tuned weights** | Pesos ajustados manualmente (sin entrenamiento). Solo posible en redes muy chicas. |
| **Arquitectura** | La estructura de la red: cuántas capas, cuántas neuronas por capa. Ej: `2-8-1`. |
| **Clamp** | Limitar un valor entre un mínimo y máximo. `max(-500, min(500, x))` = clamp a [-500, 500]. |
| **Overflow numérico** | Error cuando un número es demasiado grande para representarse. `e^10000` causa overflow. |
| **Reproductibilidad** | Capacidad de obtener los mismos resultados cada vez. Se logra con `random.seed()`. |
| **Dataset desbalanceado** | Cuando una clase tiene muchos más ejemplos que otra. Afecta cómo se mide el desempeño. |
| **Herencia de clase** | `class LayerLeaky(Layer)` crea una nueva clase que hereda todo de Layer y sobreescribe `forward`. |